# Sentiment Pipeline Demo

This notebook demonstrates the sentiment embedding pipeline from `sentiment.pipeline`.

**What it does:** news article (title + content) &rarr; BART-CNN summary &rarr; FinBERT &rarr; binary sentiment label + 768-dim embedding.

Run the cells in order. Model loading (cell 1) takes ~30 seconds on first run.

In [ ]:
from sentiment.embeddings import SentimentPipeline, aggregate_daily

pipe = SentimentPipeline()
print("Models loaded.")

In [ ]:
# Process a single article through the full pipeline
result = pipe.process_article(
    title="Apple reports record quarterly revenue",
    content=(
        "Apple Inc. reported record quarterly revenue of $124 billion for the fiscal "
        "first quarter of 2025, up 11% year over year. The company saw strong demand "
        "across its product lineup, with iPhone revenue reaching an all-time high of "
        "$71 billion. Services revenue also set a new record at $26 billion."
    ),
)

print(f"Summary:   {result['summary'][:120]}...")
print(f"Label:     {result['label']}  (1=positive, 0=negative/neutral)")
print(f"Embedding: shape={result['embedding'].shape}, dtype={result['embedding'].dtype}")

In [ ]:
import pandas as pd

# Process a batch of articles
articles = [
    {"title": "Apple reports record quarterly revenue",           "content": "Strong iPhone and services demand drove results."},
    {"title": "Apple faces regulatory scrutiny over App Store",   "content": "EU regulators opened an investigation into anti-competitive practices."},
    {"title": "Apple expands AI features across product lineup",  "content": "New AI capabilities announced for iPhone, iPad, and Mac."},
    {"title": "Apple stock dips on profit-taking after earnings", "content": "Shares fell 3% as investors took profits."},
    {"title": "Microsoft cloud revenue surges past expectations", "content": "Azure cloud revenue grew 29%, beating analyst estimates."},
    {"title": "Microsoft announces layoffs in gaming division",   "content": "1,900 employees affected following the Activision acquisition."},
]

results = pipe.process_batch(articles)

# Build a DataFrame with ticker/date metadata
tickers = ["AAPL", "AAPL", "AAPL", "AAPL", "MSFT", "MSFT"]
dates   = ["2025-01-15", "2025-01-15", "2025-01-15", "2025-01-16", "2025-01-15", "2025-01-15"]

df = pd.DataFrame([
    {"ticker": tk, "date": dt, "label": r["label"], "embedding": r["embedding"]}
    for tk, dt, r in zip(tickers, dates, results)
])

print("Per-article results:")
print(df[["ticker", "date", "label"]].to_string(index=False))

In [ ]:
# Aggregate to one row per (ticker, date) — this is what the prediction model consumes
daily = aggregate_daily(df)

print("Daily aggregated results:")
print(daily[["ticker", "date", "sentiment_score", "n_articles"]].to_string(index=False))
print()
for _, row in daily.iterrows():
    emb = row["embedding"]
    print(f"  {row['ticker']} {row['date']}: emb_shape={emb.shape}, finite={__import__('numpy').all(__import__('numpy').isfinite(emb))}")